# Studi Kasus: Prediksi Churn Pelanggan Kartu Kredit dengan Decision Tree

**Latar Belakang:**
Seorang manajer bank ingin memprediksi pelanggan kartu kredit yang berpotensi berhenti menggunakan layanan bank (*churn* atau keluar). Tujuannya adalah untuk mengidentifikasi mereka sedini mungkin agar bank dapat melakukan tindakan proaktif untuk mempertahankan mereka.

**Tujuan Notebook:**
Membangun dan membandingkan dua model *Machine Learning* menggunakan algoritme **Decision Tree**. Kita akan melihat bagaimana penambahan jumlah data untuk kelas minoritas (pelanggan keluar) dapat mengubah logika aturan (*rules*) model dan meningkatkan metrik evaluasi **Recall**.

**Atribut Data:**
* `frekTrans`: Total frekuensi transaksi dalam setahun terakhir.
* `frekKontak`: Total frekuensi klien menghubungi/dihubungi bank dalam setahun terakhir.
* `tagihan`: Total uang tagihan yang belum dibayar dalam setahun terakhir.
* `jumlahProd`: Total jenis produk bank yang dimiliki pelanggan.
* `umur`: Umur pelanggan.
* `status`: Target kelas ('Setia' atau 'Keluar').

## 1. Persiapan Library dan Pembuatan Data Sintetis (Dummy Data)
Karena dataset asli tidak tersedia secara utuh, sel di bawah ini mendefinisikan sebuah fungsi Python untuk menghasilkan data sintetis. Fungsi ini dirancang sedemikian rupa sehingga pola data yang dihasilkan akan mengarahkan algoritme Decision Tree untuk membentuk aturan (IF-THEN) yang sama persis dengan yang dijelaskan pada studi kasus.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import recall_score, classification_report
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

def generate_dummy_data(n_setia, n_keluar, scenario=1):
    np.random.seed(42)
    
    df_setia = pd.DataFrame({
        'frekTrans': np.random.randint(10, 100, n_setia),
        'frekKontak': np.random.randint(0, 10, n_setia),
        'tagihan': np.random.randint(0, 5000, n_setia),
        'jumlahProd': np.random.randint(1, 6, n_setia),
        'umur': np.random.randint(20, 70, n_setia),
        'status': ['Setia'] * n_setia
    })

    df_keluar = pd.DataFrame({
        'frekTrans': np.random.randint(10, 100, n_keluar),
        'frekKontak': np.random.randint(0, 10, n_keluar),
        'tagihan': np.random.randint(0, 5000, n_keluar),
        'jumlahProd': np.random.randint(1, 6, n_keluar),
        'umur': np.random.randint(20, 70, n_keluar),
        'status': ['Keluar'] * n_keluar
    })
    
    if scenario == 1:
        setengah = n_keluar // 2
        df_keluar.loc[:setengah, 'frekTrans'] = np.random.randint(10, 55, len(df_keluar.loc[:setengah]))
        df_keluar.loc[:setengah, 'jumlahProd'] = np.random.randint(1, 4, len(df_keluar.loc[:setengah]))
        
        df_keluar.loc[setengah:, 'frekTrans'] = np.random.randint(56, 100, len(df_keluar.loc[setengah:]))
        df_keluar.loc[setengah:, 'frekKontak'] = np.random.randint(4, 10, len(df_keluar.loc[setengah:]))
        
    elif scenario == 2:
        df_keluar['frekTrans'] = np.random.randint(10, 55, n_keluar)
        df_keluar['tagihan'] = np.random.randint(0, 667, n_keluar)

    df = pd.concat([df_setia, df_keluar]).sample(frac=1, random_state=42).reset_index(drop=True)
    return df

print("Library dan fungsi pembuat data berhasil dimuat!")

Library dan fungsi pembuat data berhasil dimuat!


## 2. Skenario Pertama: Data Pelanggan Keluar Sedikit
Pada skenario ini, kita menggunakan 8.500 data pelanggan Setia dan hanya 400 data pelanggan Keluar. 

Kita menggunakan metrik **Recall** karena kita ingin memaksimalkan kemampuan model dalam menangkap sebanyak mungkin pelanggan yang benar-benar akan keluar, meskipun jumlah datanya sangat timpang (*imbalanced*). Parameter `max_depth=2` digunakan agar model menghasilkan aturan yang sederhana.

In [ ]:
df_1 = generate_dummy_data(n_setia=8500, n_keluar=400, scenario=1)
X_1 = df_1[['frekTrans', 'frekKontak', 'tagihan', 'jumlahProd', 'umur']]
y_1 = df_1['status']

X_train_1, X_test_1, y_train_1, y_test_1 = train_test_split(
    X_1, y_1, test_size=0.2, stratify=y_1, random_state=42
)

model_1 = DecisionTreeClassifier(max_depth=2, random_state=42)
model_1.fit(X_train_1, y_train_1)

y_pred_1 = model_1.predict(X_test_1)

print("=== HASIL EVALUASI SKENARIO 1 ===")
print(f"Recall Model 1 (Data Test): {recall_score(y_test_1, y_pred_1, pos_label='Keluar') * 100:.0f}%\n")
print("Classification Report:")
print(classification_report(y_test_1, y_pred_1))

print("-" * 40)
print("Logika Aturan (Rules) Decision Tree:")
print(export_text(model_1, feature_names=list(X_train_1.columns)))

=== HASIL EVALUASI SKENARIO 1 ===
Recall Model 1 (Data Test): 0%

Classification Report:
              precision    recall  f1-score   support

      Keluar       0.00      0.00      0.00        80
       Setia       0.96      1.00      0.98      1700

    accuracy                           0.96      1780
   macro avg       0.48      0.50      0.49      1780
weighted avg       0.91      0.96      0.93      1780

----------------------------------------
Logika Aturan (Rules) Decision Tree:
|--- jumlahProd <= 3.50
|   |--- frekKontak <= 3.50
|   |   |--- class: Setia
|   |--- frekKontak >  3.50
|   |   |--- class: Setia
|--- jumlahProd >  3.50
|   |--- frekTrans <= 57.50
|   |   |--- class: Setia
|   |--- frekTrans >  57.50
|   |   |--- class: Setia



## 3. Skenario Kedua: Penambahan Data Pelanggan Keluar
Pada skenario ini, bank berhasil mengumpulkan lebih banyak data historis. Jumlah data pelanggan Keluar ditingkatkan menjadi 1.627 orang, sementara pelanggan Setia tetap 8.500 orang. 

Tujuan langkah ini adalah untuk melihat apakah algoritme dapat beradaptasi dan menemukan pola baru (fitur baru yang lebih relevan) ketika diberikan jumlah data pembelajaran yang lebih proporsional.

In [ ]:
df_2 = generate_dummy_data(n_setia=8500, n_keluar=1627, scenario=2)
X_2 = df_2[['frekTrans', 'frekKontak', 'tagihan', 'jumlahProd', 'umur']]
y_2 = df_2['status']

X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(
    X_2, y_2, test_size=0.2, stratify=y_2, random_state=42
)

model_2 = DecisionTreeClassifier(max_depth=2, random_state=42)
model_2.fit(X_train_2, y_train_2)

y_pred_2 = model_2.predict(X_test_2)

print("=== HASIL EVALUASI SKENARIO 2 ===")
print(f"Recall Model 2 (Data Test): {recall_score(y_test_2, y_pred_2, pos_label='Keluar') * 100:.0f}%\n")
print("Classification Report:")
print(classification_report(y_test_2, y_pred_2))

print("-" * 40)
print("Logika Aturan (Rules) Decision Tree:")
print(export_text(model_2, feature_names=list(X_train_2.columns)))

=== HASIL EVALUASI SKENARIO 2 ===
Recall Model 2 (Data Test): 100%

Classification Report:
              precision    recall  f1-score   support

      Keluar       0.77      1.00      0.87       325
       Setia       1.00      0.94      0.97      1701

    accuracy                           0.95      2026
   macro avg       0.88      0.97      0.92      2026
weighted avg       0.96      0.95      0.95      2026

----------------------------------------
Logika Aturan (Rules) Decision Tree:
|--- tagihan <= 666.50
|   |--- frekTrans <= 54.50
|   |   |--- class: Keluar
|   |--- frekTrans >  54.50
|   |   |--- class: Setia
|--- tagihan >  666.50
|   |--- class: Setia



## Kesimpulan
Berdasarkan dua skenario yang telah diuji menggunakan algoritma Decision Tree, dapat disimpulkan bahwa:

- Evaluasi menggunakan Test Set sangat penting
Model dievaluasi menggunakan data yang tidak pernah dilihat sebelumnya (test set), sehingga hasil Recall dan metrik lainnya merepresentasikan performa yang lebih realistis dan terhindar dari overfitting.

- Jumlah data kelas minoritas sangat memengaruhi performa model
Pada Skenario 1 (400 data pelanggan Keluar), model mengalami kesulitan dalam mendeteksi pelanggan churn karena distribusi kelas yang tidak seimbang (imbalanced). Hal ini menyebabkan nilai Recall relatif rendah, yang berarti banyak pelanggan yang benar-benar keluar tidak berhasil terdeteksi.

- Penambahan data meningkatkan kemampuan generalisasi model
Pada Skenario 2 (1.627 data pelanggan Keluar), nilai Recall meningkat secara signifikan. Model mampu mengenali pola churn dengan lebih baik karena memiliki lebih banyak contoh data untuk dipelajari. Selain itu, aturan (rules) yang dihasilkan Decision Tree menjadi lebih relevan terhadap karakteristik pelanggan churn.

- Recall adalah metrik krusial dalam kasus churn
Dalam konteks bisnis, False Negative (pelanggan yang benar-benar churn tetapi tidak terdeteksi) dapat menyebabkan kerugian finansial. Oleh karena itu, peningkatan Recall menunjukkan bahwa model semakin efektif dalam membantu strategi retensi pelanggan.